#**In LangChain, the Agent can access many dataframes at once**

#**01: Install All the Required Packages**

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-google-genai python-dotenv
%pip install -U langchain-experimental

In [9]:
%pip install tabulate

  Obtaining dependency information for tabulate from https://files.pythonhosted.org/packages/40/44/4a5f08c96eb108af5cb50b41f76142f0afa346dfa99d5296fe7202a11854/tabulate-0.9.0-py3-none-any.whl.metadata
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python312\\Scripts\\tabulate.exe' -> 'c:\\Python312\\Scripts\\tabulate.exe.deleteme'


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#**02: Import All The Required Libraries**

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd

#**03: Setup the Environment**


In [4]:
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is not set in the .env file.")
os.environ["GOOGLE_API_KEY"] = google_api_key

#**04:Data**

In [5]:
url = "https://raw.githubusercontent.com/adamerose/datasets/master/titanic.csv"
df=pd.read_csv(url)
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


#**05: Single DataFrame Example**

#**Agent can interact with Single DataFrame**

In [6]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [7]:
agent=create_pandas_dataframe_agent(llm, df, verbose=True, allow_dangerous_code=True)

In [8]:
agent.run("How many rows are there")

C:\Users\bibhu\AppData\Local\Temp\ipykernel_25044\1005132620.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  agent.run("How many rows are there")




> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input: len(df)891I now know the final answer
Final Answer: 891

> Finished chain.


'891'

In [9]:
agent.run("How many people have more than 23 age")



> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input: print(df[df['age'] > 23].shape[0])468
I now know the final answer
Final Answer: 468

> Finished chain.


'468'

#**06: Multi DataFrame Example**

#**Agent can also interact with Multiple DataFrames passed in a list**

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [11]:
df1=df.copy()

In [12]:
df1

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [13]:
df1["age"] = df1["age"].fillna(df1["age"].mean())

In [14]:
agent=create_pandas_dataframe_agent(llm, [df, df1], verbose=True,allow_dangerous_code=True)

In [16]:
agent.invoke({"input": "How many rows in the age column are different?"})
# agent.invoke({"input": "Compare the age column in both dataframes and give me the summary of differences."})



> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input:
print(sum(df1['age'] != df2['age']))177
I now know the final answer
Final Answer: There are 177 rows in the age column that are different.

> Finished chain.


{'input': 'How many rows in the age column are different?',
 'output': 'There are 177 rows in the age column that are different.'}

In [17]:
df2=df1.copy()

In [18]:
df2["Age_Multiplied"]=df1["age"] * 2

In [19]:
df2.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,Age_Multiplied
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,44.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,76.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,52.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,70.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,70.0


In [20]:
agent=create_pandas_dataframe_agent(llm, [df,df1, df2], verbose=True, allow_dangerous_code=True)

In [22]:
agent.run("Are the number of columns same in all the dataframe")



> Entering new AgentExecutor chain...
Action: python_repl_ast
Action Input: print(len(df1.columns))15
Action: python_repl_ast
Action Input: print(len(df2.columns))15
Action: python_repl_ast
Action Input: print(len(df3.columns))16
I can see from the previous observations that df1 and df2 have 15 columns, while df3 has 16 columns. Therefore, the number of columns is not the same in all dataframes.
Final Answer: No, the number of columns is not the same in all dataframes. df1 and df2 have 15 columns, while df3 has 16 columns.

> Finished chain.


'No, the number of columns is not the same in all dataframes. df1 and df2 have 15 columns, while df3 has 16 columns.'